# CE25M108_Test-2

## Question 1: Interactive Folium Map of Waste-Bin Observations 

In [3]:

import geopandas as gpd
import pandas as pd
import sqlite3
import folium
import shutil
import os

# ---- Step 1: Load waste-bin points from the gpkg ----
wastebins = gpd.read_file("CE25M108.gpkg", layer="notes_point")
wastebins = wastebins.to_crs(epsg=4326)   # ensure lat/lon for Folium

print("Number of waste-bin observations:", len(wastebins))
display(wastebins.head())

Number of waste-bin observations: 40


,uuid,color,title,note,timestamp,geometry
0,{f1b88627-ed7e-44e6-bde9-e7628ee8d544},#377eb8,Malhar hostel Entry Gate,Full Bin,2026-09-06 03:44:20.073000+00:00,POINT Z (79.58745 13.71772 0)
1,{09a5a1d4-6c82-47f7-b3c5-c476a03b663b},#377eb8,North mess entry Gate,Overflowing Bin,2026-09-06 04:09:53.513000+00:00,POINT Z (79.58704 13.71706 0)
2,{e368f6fc-6df4-4bd0-b7ad-4b6dea245b43},#377eb8,Near the Malhar and Des hostel,Half Bin,2026-09-06 04:21:41.154000+00:00,POINT Z (79.58713 13.71798 0)
3,{923de2a6-1e57-439e-aa47-e387aca30f22},#377eb8,Infront of Des hostel,Overflowing Bin,2026-09-06 04:27:30.753000+00:00,POINT Z (79.58694 13.71758 0)
4,{ab50ce14-d880-48a9-a6c1-57a62ad5b5d4},#377eb8,Des hostel entry Gate,Full Bin,2026-09-06 04:29:18.832000+00:00,POINT Z (79.58673 13.7177 0)


### Step 2: Load the attachments table (photo file links) from the same gpkg

In [4]:
con = sqlite3.connect("CE25M108.gpkg")
attachments = pd.read_sql_query("SELECT note_uuid, media FROM notes_attachments", con)
con.close()

# Keep only the filename (not the full DCIM/... path)
attachments["photo_filename"] = attachments["media"].apply(lambda x: os.path.basename(x))

print("Number of attachments:", len(attachments))
display(attachments.head())

Number of attachments: 40


,note_uuid,media,photo_filename
0,{f1b88627-ed7e-44e6-bde9-e7628ee8d544},DCIM/JPEG_20260906034440526.jpg,JPEG_20260906034440526.jpg
1,{09a5a1d4-6c82-47f7-b3c5-c476a03b663b},DCIM/JPEG_20260906041012231.jpg,JPEG_20260906041012231.jpg
2,{e368f6fc-6df4-4bd0-b7ad-4b6dea245b43},DCIM/JPEG_20260906042200149.jpg,JPEG_20260906042200149.jpg
3,{923de2a6-1e57-439e-aa47-e387aca30f22},DCIM/JPEG_20260906042744573.jpg,JPEG_20260906042744573.jpg
4,{ab50ce14-d880-48a9-a6c1-57a62ad5b5d4},DCIM/JPEG_20260906042929993.jpg,JPEG_20260906042929993.jpg


### Step 3: Merge photo filenames onto the waste-bin points (match on uuid) 

In [5]:
wastebins = wastebins.merge(
    attachments[["note_uuid", "photo_filename"]],
    left_on="uuid", right_on="note_uuid", how="left"
)

print("Points with a matched photo:", wastebins["photo_filename"].notna().sum())

Points with a matched photo: 40


### Step 4: Prepare a local "photos" folder next to the notebook

In [6]:
os.makedirs("photos", exist_ok=True)

In [11]:
import os

photo_lookup = {}
for root, dirs, files in os.walk("photos_extracted"):
    for f in files:
        if f.lower().endswith((".jpg", ".jpeg")):
            photo_lookup[f] = os.path.join(root, f)

print("Total JPGs mili:", len(photo_lookup))

Total JPGs mili: 40


### Step 5: Build the Folium map 

In [12]:
import os, io, base64
import pandas as pd
import folium
from PIL import Image, ImageOps
from folium.plugins import Fullscreen

def photo_to_base64(path, max_px=640, quality=70):
    img = Image.open(path)
    img = ImageOps.exif_transpose(img).convert("RGB")
    img.thumbnail((max_px, max_px))
    buf = io.BytesIO()
    img.save(buf, format="JPEG", quality=quality, optimize=True)
    return base64.b64encode(buf.getvalue()).decode("utf-8")


center_lat = wastebins.geometry.y.mean()
center_lon = wastebins.geometry.x.mean()

m = folium.Map(location=[center_lat, center_lon], zoom_start=17, max_zoom=19, tiles=None)

folium.TileLayer(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
    attr="Esri World Imagery", name="Satellite (Esri)",
    overlay=False, control=True, max_zoom=19, show=True
).add_to(m)

folium.TileLayer(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Street_Map/MapServer/tile/{z}/{y}/{x}",
    attr="Esri World Street Map", name="Street Map (Esri)",
    overlay=False, control=True, max_zoom=19, show=False
).add_to(m)


COLLECTED_BY = "Manish Chouhan (CE25M108)"
fg = folium.FeatureGroup(name="Waste Bins - Manish Chouhan (CE25M108)")

missing = 0
for idx, row in wastebins.iterrows():
    photo_html = ""
    photo_path = photo_lookup.get(row["photo_filename"])
    if photo_path:
        b64 = photo_to_base64(photo_path)
        photo_html = (f'<br><img src="data:image/jpeg;base64,{b64}" width="240" '
                      f'style="border-radius:6px; margin-top:6px;">')
    else:
        missing += 1

    popup_html = f"""
    <div style="font-family: Arial; font-size: 13px;">
        <b style="font-size:14px;">{row['title']}</b><br>
        <b>Status / Note:</b> {row['note']}<br>
        <b>Collected by:</b> {COLLECTED_BY}<br>
        <b>Timestamp:</b> {row['timestamp']}<br>
        <b>Coordinates:</b> {row.geometry.y:.6f}, {row.geometry.x:.6f}
        {photo_html}
    </div>"""

    folium.Marker(
        location=[row.geometry.y, row.geometry.x],
        tooltip=f"{row['title']} | {row['note']}",
        popup=folium.Popup(popup_html, max_width=300),
        icon=folium.Icon(color="green", icon="trash", prefix="fa")
    ).add_to(fg)

fg.add_to(m)
folium.LayerControl(collapsed=False).add_to(m)
Fullscreen().add_to(m)


m.save("CE25M108_wastebin_map.html")
size_mb = os.path.getsize("CE25M108_wastebin_map.html") / 1e6
print(f"Saved. File size: {size_mb:.1f} MB | photos not found: {missing}")

Saved. File size: 2.4 MB | photos not found: 0


### Step 6: Save the map and copy photos next to it (for hosting

In [13]:
m.save("CE25M108_wastebin_map.html")

print("Map saved as CE25M108_wastebin_map.html")
print("Make sure the 'photos' folder is uploaded alongside this HTML file when hosting.")

Map saved as CE25M108_wastebin_map.html
Make sure the 'photos' folder is uploaded alongside this HTML file when hosting.
